# Модуль 6. Data Leakage в препроцессинге и защита через Pipeline

**Длительность:** 90 минут
**Формат:** теория (65 мин) + практика (25 мин)
**Цель модуля:** Учащийся должен понимать точный механизм, по которому «безобидные» шаги препроцессинга (масштабирование, заполнение пропусков, кодирование категорий) создают утечку данных при неверном порядке операций, уметь формально обосновать правило `fit` только на train, и уметь построить единственную архитектурно надежную защиту от этого класса ошибок — `Pipeline` в связке с `ColumnTransformer`.

## 1. Введение: ещё один источник утечки (5 мин)

### 1.1. Место этого модуля в общей картине
Модуль 5 разобрал утечку данных, возникающую из-за неверного порядка операций с SMOTE. Это частный случай куда более общей и куда более распространенной проблемы: **любой шаг препроцессинга, параметры которого вычисляются по данным (среднее, дисперсия, медиана, набор уникальных категорий), способен утечь информацию из валидации в обучение**, если применен в неверном порядке относительно разбиения на train/test.

### 1.2. Почему это самая частая ошибка на практике
В отличие от SMOTE (который применяется не всегда), масштабирование признаков и заполнение пропусков используются почти в каждом пайплайне классического машинного обучения. Это делает связку «`StandardScaler().fit_transform(X)` на всей выборке, затем `train_test_split`» одной из самых массовых методологических ошибок в индустрии — настолько массовой, что именно с неё принято начинать разбор Data Leakage в большинстве серьезных курсов по ML-инженерии.

## 2. Классическая ошибка: `fit_transform` до `split` — математика утечки (20 мин)

### 2.1. Формулировка ошибки в коде

In [ ]:
# НЕПРАВИЛЬНО
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)          # параметры считаются по ВСЕМУ X
X_train, X_test, y_train, y_test = train_test_split(X_scaled, y, test_size=0.3)

Выглядит невинно: код синтаксически корректен, ничего не падает, метрики после обучения выглядят разумно. Проблема — не в синтаксисе, а в порядке операций.

### 2.2. Что именно вычисляет `StandardScaler`
$$z = \frac{x - \mu}{\sigma}$$
где $\mu$ и $\sigma$ — среднее и стандартное отклонение, вычисленные методом `.fit()` по переданным ему данным. Если `.fit()` вызван на полном $X$ (до разбиения), то $\mu$ и $\sigma$ являются функцией **в том числе** от значений, которые впоследствии окажутся в тестовой выборке.

### 2.3. В чем именно состоит утечка
Утечка здесь — не в том, что модель «увидела» сырые тестовые данные (она их действительно не видела). Утечка тоньше: **сама система координат, в которой представлены обучающие данные, была откалибрована с учетом информации о тестовых данных.** Каждое обучающее наблюдение $x_i^{train}$ после трансформации $\frac{x_i^{train}-\mu}{\sigma}$ несет в себе отпечаток $\mu$ и $\sigma$ — величин, вычисленных с участием тестовых точек. Модель, обучаясь на таких признаках, неявно «знает» немного больше о распределении данных в целом (включая тест), чем должна бы знать по методологии.

### 2.4. Числовой пример: как сильно это меняет обучающие признаки
Рассмотрим крошечный, но показательный пример. Есть 6 значений одного признака, из которых первые 4 попадут в train, а последние 2 — в test:
$$X = [10,\ 12,\ 11,\ 13,\ \underbrace{100,\ 105}_{\text{будущий test}}]$$

**Неправильный расчет ($\mu, \sigma$ по всем 6 значениям):**
$$\mu_{\text{all}} = \frac{10+12+11+13+100+105}{6} = \frac{251}{6} \approx 41.83$$
$$\sigma_{\text{all}} = \sqrt{\frac{1}{6}\sum(x_i-\mu_{\text{all}})^2} \approx 42.93$$

**Правильный расчет ($\mu, \sigma$ только по 4 train-значениям):**
$$\mu_{\text{train}} = \frac{10+12+11+13}{4} = 11.5$$
$$\sigma_{\text{train}} = \sqrt{\frac{1}{4}\sum(x_i-\mu_{\text{train}})^2} \approx 1.118$$

Посчитаем, во что превращается **одно и то же** обучающее значение $x=10$ в двух вариантах:

| Способ расчета | Формула | Результат |
|---|---|---|
| Неправильный ($\mu_{\text{all}}, \sigma_{\text{all}}$) | $(10 - 41.83)/42.93$ | $\approx -0.741$ |
| Правильный ($\mu_{\text{train}}, \sigma_{\text{train}}$) | $(10 - 11.5)/1.118$ | $\approx -1.342$ |

Разница не косметическая: $\sigma_{\text{all}}$ оказался почти в **38 раз больше** ($42.93$ против $1.118$), потому что на него повлиял разброс, привнесенный будущими тестовыми значениями $100$ и $105$. В результате все обучающие признаки оказываются искусственно «сжаты» к нулю по сравнению с тем, какими они выглядели бы, если бы модель обучалась, ничего не зная о существовании тестовых выбросов. Модель обучается на данных, которые выглядят иначе, чем должны бы выглядеть при честной методологии — именно это и называется утечкой.

## 3. Насколько велик этот эффект на практике? (10 мин)

### 3.1. Не всегда драматично — но всегда неправильно
В отличие от утечки через SMOTE (Модуль 5), где эффект на метрику часто заметен невооруженным глазом, утечка через простое масштабирование или импутацию медианой на **больших** датасетах обычно дает **небольшое** искажение $\mu, \sigma$ — по закону больших чисел среднее по $n$ наблюдениям и среднее по $0.8n$ наблюдениям (после отбрасывания 20% на тест) сходятся друг к другу при росте $n$. Именно поэтому эта ошибка часто «сходит с рук» на больших датасетах: метрики почти не искажаются, и проблема остается незамеченной.

### 3.2. Когда эффект становится значительным

Эффект усиливается в нескольких характерных ситуациях:
- **Малый объем данных** — как в числовом примере раздела 2.4, где всего 2 «тестовые» точки радикально сдвинули $\sigma$.
- **Импутация, а не масштабирование.** Замена пропусков медианой/модой, посчитанной по всем данным (включая тест), может систематически сдвигать значения признака у объектов с пропусками, причем этот сдвиг напрямую входит в само значение признака (а не просто в масштаб), что потенциально сильнее влияет на решающую границу модели, чем сдвиг $\mu,\sigma$ при масштабировании.
- **One-Hot Encoding редких категорий.** Если кодирование категориального признака выполнено на полном датасете, набор столбцов результирующей матрицы фиксируется с учетом категорий, которые встречаются только в тестовой части. Это не приводит к прямой утечке чисел (обучающие строки по-прежнему имеют ноль в этих столбцах), но является симптомом той же методологической ошибки и создает **отдельный** практический риск — раздел 3.3.

### 3.3. Смежная, но принципиально иная проблема: незнакомые категории при верном порядке операций
Если, наоборот, обучить `OneHotEncoder` **правильно** (только на train), возникает законный инженерный вопрос: что делать, если в тестовых (или боевых, продакшен) данных встретится категория, которой не было в обучающей выборке? Без специальной обработки `.transform()` на неизвестной категории выбросит `ValueError`. Это **не утечка**, а обратная, инженерная проблема — и решается она параметром `handle_unknown='ignore'` (энкодер кодирует незнакомую категорию как вектор из нулей по всем известным столбцам, не вызывая исключения). Важно не путать эти два разных вопроса на собеседовании: «утечка происходит, если фитить на данных, которые не должны быть видны на этапе обучения» — это про раздел 2–3; «что делать с категориями, которых не было в train» — это отдельный вопрос о надежности инференса, не имеющий отношения к валидности метрик.

## 4. Формальное правило: `fit` только на train, `transform` на train и test (10 мин)

### 4.1. Правило
$$\text{scaler.fit(X\_train)} \;\rightarrow\; \text{scaler.transform(X\_train)}, \ \text{scaler.transform(X\_test)}$$
Параметры вычисляются **один раз**, только по обучающим данным. Затем эти же зафиксированные параметры применяются (не пересчитываются!) и к train, и к test.

### 4.2. Это общий принцип, а не частность про `StandardScaler`
Каждый Transformer в Scikit-Learn, запоминающий что-либо на этапе `.fit()`, обязан подчиняться этому правилу:

| Transformer | Что запоминает `.fit()` |
|---|---|
| `StandardScaler` | среднее $\mu$, стандартное отклонение $\sigma$ |
| `MinMaxScaler` | минимум и максимум признака |
| `SimpleImputer` | медиану / среднее / моду для заполнения пропусков |
| `OneHotEncoder` | список уникальных категорий и порядок соответствующих им столбцов |
| `PolynomialFeatures` | степени и комбинации признаков (структурный параметр, не зависит от данных, но для полноты включен как пример другого рода шага) |

Любой из этих объектов, если вызвать его `.fit()` (или `.fit_transform()`) на полном датасете до разбиения, воспроизведет ту же категорию ошибки, что разобрана в разделе 2, просто с другой конкретной величиной, которая «утекает».

### 4.3. Почему интерфейс `fit`/`transform` устроен именно так
Разделение `.fit()` и `.transform()` на два отдельных метода в API Scikit-Learn — не случайность дизайна, а прямое отражение методологического требования: `.fit()` — это единственное место, где объект имеет право «смотреть» на данные и извлекать из них статистики; `.transform()` — чисто механическое применение уже зафиксированных чисел, не требующее (и не имеющее права требовать) никакой новой информации о данных, к которым применяется.

## 5. `ColumnTransformer`: раздельная обработка числовых и категориальных признаков (15 мин)

### 5.1. Проблема разнородных данных
Реальные датасеты почти никогда не состоят из одних только числовых признаков. Числовые признаки нуждаются в масштабировании и, возможно, заполнении пропусков числом; категориальные — в кодировании и, возможно, заполнении пропусков модой (самой частой категорией). Применение одного и того же `Pipeline` ко всем столбцам сразу невозможно — `StandardScaler`, например, упадет на строковой колонке.

### 5.2. Механика `ColumnTransformer`

In [ ]:
from sklearn.compose import ColumnTransformer

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),        # применяется только к numeric_features
    ('cat', categorical_pipeline, categorical_features)  # применяется только к categorical_features
])

`ColumnTransformer` принимает список троек `(имя, трансформер, список_колонок)`. На `.fit()` каждый трансформер обучается **только** на своем подмножестве колонок; на `.transform()` каждый трансформер применяется к своему подмножеству, а результаты склеиваются в одну общую матрицу (`np.hstack` для плотных массивов, `scipy.sparse.hstack` при наличии разреженных выходов, например от `OneHotEncoder`).

### 5.3. Важная оговорка: `ColumnTransformer` сам по себе не защищает от утечки
`ColumnTransformer` — это инструмент для **разделения** обработки по типам колонок, а не для защиты от утечки как таковой. Утечка (раздел 2) произойдет ровно так же, если вызвать `preprocessor.fit_transform(X)` на всем датасете до разбиения — `ColumnTransformer` унаследует ту же проблему от вложенных в него трансформеров. Настоящая защита обеспечивается только связкой с `Pipeline` и корректным местом вызова `.fit()`, что разбирается в следующем разделе.

## 6. `Pipeline`: инкапсуляция как единственная надежная защита (15 мин)

### 6.1. Идея
`Pipeline` объединяет `ColumnTransformer` (и любые другие шаги препроцессинга) и финальную модель в единый объект с одним общим интерфейсом `.fit()` / `.predict()`:

In [ ]:
full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression())
])

### 6.2. Почему это архитектурно, а не дисциплинарно надежно
Ключевой механизм — то, что происходит внутри `cross_val_score(full_pipeline, X, y, cv=...)` или внутри `GridSearchCV`. На каждом сплите кросс-валидации библиотека вызывает:
$$\text{full\_pipeline.fit(X\_train\_fold,\ y\_train\_fold)}$$
Вызов `.fit()` на `Pipeline` **каскадно** вызывает `.fit_transform()` для каждого промежуточного шага **строго на переданных ему данных**, то есть только на `X_train_fold` текущего сплита. Затем для оценки на валидационной части того же сплита вызывается:
$$\text{full\_pipeline.predict(X\_val\_fold)}$$
На этом вызове каждый промежуточный шаг применяет уже зафиксированные на предыдущем шаге параметры через `.transform()`, не пересчитывая их.

**Из этого следует важнейшее практическое свойство:** для **каждого** фолда кросс-валидации `Pipeline` автоматически, без единой лишней строчки кода, обучает свой собственный `StandardScaler`, свой собственный `SimpleImputer`, свой собственный `OneHotEncoder` — исключительно по обучающей части именно этого фолда. Программисту физически негде допустить ошибку раздела 2 — она структурно исключена самим устройством вызовов `cross_val_score`/`GridSearchCV` над объектом `Pipeline`.

### 6.3. Параллель с Модулем 5
Это прямое обобщение идеи из Модуля 5 про `imblearn.pipeline.Pipeline`: там sampler-шаг (SMOTE) был активен только на `.fit()` и пропускался на `.predict()`/`.transform()`. Здесь тот же структурный принцип реализован иначе (обычные Transformer-ы всегда и обучаются, и применяются, но обучаются **только** на той части данных, которая передана в конкретный вызов `.fit()`), но итоговый защитный эффект — идентичен: правильное поведение гарантируется архитектурой, а не памятью и аккуратностью разработчика.

### 6.4. Почему «дисциплинарная» защита (просто не забывать делать по правилам) — ненадежна
Ручной контроль порядка операций (написать код так, чтобы `.fit()` scaler-а вызывался строго после `train_test_split`) в теории работает, но на практике не масштабируется: при добавлении кросс-валидации нужно переобучать препроцессор заново на **каждом** фолде вручную (что легко забыть сделать при рефакторинге кода), при добавлении `GridSearchCV` — на каждой комбинации гиперпараметров и каждом фолде одновременно. `Pipeline` делает это автоматически и единообразно в любом сценарии, что и объясняет фразу «единственная надежная защита» в названии раздела курса.

## 7. Практика: «грязный» и «чистый» пайплайн на одних данных (25 мин)

### 7.1. Постановка задачи
Сгенерировать датасет со смешанными типами признаков (числовые с пропусками + категориальный), реализовать оба подхода — «грязный» (препроцессинг единожды на всех данных до кросс-валидации) и «чистый» (препроцессинг внутри `Pipeline`, пересчитываемый на каждом фолде) — **на одних и тех же фолдах StratifiedKFold**, чтобы сравнение было честным и парным, и сопоставить LogLoss.

### 7.2. Код: генерация данных

In [ ]:
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedKFold, cross_val_score
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.impute import SimpleImputer
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.linear_model import LogisticRegression

np.random.seed(42)
n = 400   # намеренно небольшой датасет — эффект утечки от масштабирования заметнее
          # на малых выборках (раздел 3.1); на больших датасетах эффект был бы,
          # но гораздо более скромным по величине

numeric_1 = np.random.randn(n) * 10 + 50
numeric_2 = np.random.exponential(scale=5, size=n)
category = np.random.choice(['A', 'B', 'C', 'D'], size=n, p=[0.4, 0.3, 0.2, 0.1])

# Пропуски — 10% случайных значений в каждом числовом признаке
numeric_1[np.random.rand(n) < 0.10] = np.nan
numeric_2[np.random.rand(n) < 0.10] = np.nan

# Целевая переменная, реально зависящая от признаков
logit = 0.05 * (np.nan_to_num(numeric_1, nan=50) - 50) \
        + 0.15 * np.nan_to_num(numeric_2, nan=5) \
        + (category == 'A').astype(float) * 0.6
prob = 1 / (1 + np.exp(-logit))
y = (np.random.rand(n) < prob).astype(int)

df = pd.DataFrame({'numeric_1': numeric_1, 'numeric_2': numeric_2, 'category': category})
numeric_features = ['numeric_1', 'numeric_2']
categorical_features = ['category']

### 7.3. Код: чистый подход (Pipeline внутри cross_val_score)

In [ ]:
numeric_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='median')),
    ('scaler', StandardScaler())
])

categorical_pipeline = Pipeline([
    ('imputer', SimpleImputer(strategy='most_frequent')),
    ('encoder', OneHotEncoder(handle_unknown='ignore', drop='first'))
])

preprocessor = ColumnTransformer([
    ('num', numeric_pipeline, numeric_features),
    ('cat', categorical_pipeline, categorical_features)
])

full_pipeline = Pipeline([
    ('preprocessor', preprocessor),
    ('model', LogisticRegression(max_iter=1000, random_state=42))
])

skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scores_clean = cross_val_score(full_pipeline, df, y, cv=skf, scoring='neg_log_loss')

print(f"LogLoss по фолдам (чистый подход): {np.round(-scores_clean, 5)}")
print(f"Средний LogLoss (чистый подход):   {-scores_clean.mean():.5f} ± {scores_clean.std():.5f}")

### 7.4. Код: грязный подход (препроцессинг единожды на всех данных)

In [ ]:
# Препроцессинг единожды на ВСЕХ данных, ДО того как cross_val_score начнет
# делить их на фолды. Каждый шаг видит информацию обо всём датасете сразу.
num_imputer_dirty = SimpleImputer(strategy='median')
X_num_dirty = num_imputer_dirty.fit_transform(df[numeric_features])

scaler_dirty = StandardScaler()
X_num_dirty = scaler_dirty.fit_transform(X_num_dirty)

cat_imputer_dirty = SimpleImputer(strategy='most_frequent')
X_cat_dirty = cat_imputer_dirty.fit_transform(df[categorical_features])

encoder_dirty = OneHotEncoder(sparse_output=False, drop='first')
X_cat_dirty = encoder_dirty.fit_transform(X_cat_dirty)

X_dirty_full = np.hstack([X_num_dirty, X_cat_dirty])

# Используем ТЕ ЖЕ САМЫЕ фолды skf — сравнение честное и парное
scores_dirty = cross_val_score(
    LogisticRegression(max_iter=1000, random_state=42),
    X_dirty_full, y, cv=skf, scoring='neg_log_loss'
)

print(f"\nLogLoss по фолдам (грязный подход): {np.round(-scores_dirty, 5)}")
print(f"Средний LogLoss (грязный подход):   {-scores_dirty.mean():.5f} ± {scores_dirty.std():.5f}")

### 7.5. Код: итоговое сравнение

In [ ]:
print(f"\n--- Итог ---")
print(f"Грязный подход (с утечкой):     LogLoss = {-scores_dirty.mean():.5f}")
print(f"Чистый подход (без утечки):     LogLoss = {-scores_clean.mean():.5f}")
print(f"Разница:                        {(-scores_dirty.mean()) - (-scores_clean.mean()):+.5f}")

(Напомним: **меньший** LogLoss — лучше; значит, оптимистичная (то есть заниженная) оценка LogLoss у «грязного» подхода будет говорить о завышенной, недостоверной оценке качества.)

### 7.6. Что должно получиться и как интерпретировать
- LogLoss «грязного» подхода должен оказаться **немного ниже** (оптимистичнее), чем у «чистого» — потому что признаки в грязном варианте отшкалированы с использованием информации, включающей каждый конкретный валидационный фолд, что делает задачу предсказания искусственно немного легче для модели на этом фолде.
- Разница будет **гораздо скромнее**, чем аналогичная демонстрация утечки через SMOTE в Модуле 5 — это ожидаемо и согласуется с разделом 3.1: простое масштабирование и импутация медианой искажают данные слабее, чем генерация синтетических почти-дубликатов. Тем не менее направление эффекта (грязный подход оптимистичнее чистого) должно быть устойчиво воспроизводимо.
- Полезное самостоятельное упражнение: увеличить `n` с 400 до, например, 20 000 и перезапустить сравнение — разница между подходами должна заметно сократиться, эмпирически подтверждая рассуждение раздела 3.1 о сходимости $\mu, \sigma$ по закону больших чисел.

## 8. Итоги модуля (5 мин)

### Ключевые тезисы
1. Утечка через препроцессинг возникает всякий раз, когда `.fit()` любого трансформера (scaler, imputer, encoder) вызывается на данных, включающих то, что впоследствии станет валидацией — параметры трансформации оказываются функцией от тестовых значений.
2. Эффект утечки от простого масштабирования/импутации обычно скромный на больших датасетах (закон больших чисел), но методологически неверен всегда, независимо от размера данных, и может стать значительным на малых выборках или при использовании более «жадных» до информации шагов (импутация медианой, работа с редкими категориями).
3. Формальное правило: `.fit()` — только на train; `.transform()` — на train и на test, без повторного пересчета параметров.
4. `ColumnTransformer` разделяет обработку числовых и категориальных признаков внутри одного объекта, но сам по себе не защищает от утечки — защита обеспечивается только правильным местом вызова `.fit()`.
5. `Pipeline` внутри `cross_val_score`/`GridSearchCV` структурно исключает утечку: на каждом фолде препроцессор переобучается заново, используя только обучающую часть этого конкретного фолда — программист не может забыть это сделать, потому что это делает библиотека.

### Контрольные вопросы
- Почему утечка через `StandardScaler().fit_transform(X)` на полном датасете не связана напрямую с тем, что модель «увидела» тестовые значения признаков?
- Почему эффект такой утечки обычно тем заметнее, чем меньше размер датасета?
- В чем разница между проблемой утечки при `fit_transform` на всем `X` и проблемой обработки незнакомых категорий при `handle_unknown='ignore'`? Почему это не одна и та же проблема?
- Опишите пошагово, что происходит внутри `cross_val_score(full_pipeline, X, y, cv=5)` на уровне вызовов `.fit()` и `.predict()`, и почему это гарантирует отсутствие утечки без дополнительных действий со стороны программиста.
- Почему `ColumnTransformer` сам по себе не является защитой от Data Leakage?

### Что дальше
Модули 1–6 закрыли методологию честной валидации и честных метрик на исторических данных. В следующем модуле мы переходим к вопросу, что происходит с моделью **после** выкладки в продакшен, когда истинные метки $y$ приходят с задержкой (или не приходят вовсе) — как обнаружить деградацию модели по одним только входным признакам, до того как бизнес почувствует падение качества на своих деньгах.